# <center>  **<span style="font-size:80px;">Exploración de los Datos</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os


from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, classification_report
)


sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys
from src.config import Paths
Paths.init_project()

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor
preprocessor = WaterPreprocessor()

In [ ]:
seed = 80

images_path = Path("./EXTERNAL")
images_path.mkdir(parents=True, exist_ok=True)

# AMAEM

In [ ]:
df_amaem = pd.read_csv(Paths.PROC_CSV_AMAEM)
df_amaem[DatasetKeys.FECHA] = pd.to_datetime(df_amaem[DatasetKeys.FECHA])

# Instituto Nacional de Estadística (INE)

In [ ]:
def process_df(df, mapeo=None, drop=None):
    df.columns = (df.columns
                  .str.strip().str.lower()
                  .str.replace(' ', '_').str.replace(':', '')
                  .str.replace('á', 'a').str.replace('é', 'e')
                  .str.replace('í', 'i').str.replace('ó', 'o')
                  .str.replace('ú', 'u'))
    
   
    if mapeo:
        df = df.rename(columns=mapeo)
    
    if drop:
        df = df.drop(columns=drop)
        
    return df

## Municipios

In [ ]:
params_mun = {"encoding": "latin1", "sep": "\t"}

# Cargar archivos de Municipios
df_municipios_plazas   = pd.read_csv(Paths.INE_MUNICIPIOS_PLAZAS, **params_mun)
df_municipios_porcent  = pd.read_csv(Paths.INE_MUNICIPIOS_PORCENT, **params_mun)

### Matching Barrios - Municipios

Nuestro datos *INE* catalogan los datos a través de **municipios**. En cambio, nuestros datos *AMAEM* utilizan los **barrios**. Por tanto, deberemos asignar a cada barrio su correspondiente municipio. En caso de que no se encuentre el municipio exacto utilizaremos una ponderación.

In [ ]:
df_amaem.barrio.unique()

In [ ]:
df_municipios_porcent.Municipios.unique()

## Provincia

In [ ]:
params_prov = {"encoding": "utf-8", "sep": ";"}

# Cargar archivos de Provincia
df_provincia_hotel     = pd.read_csv(Paths.INE_PROVINCIA_HOTEL, **params_prov)
df_provincia_vt        = pd.read_csv(Paths.INE_PROVINCIA_VT, **params_prov)


### Viviendas Trurísticas

In [ ]:
mapeo_prov = {
    "Fecha": DatasetKeys.FECHA,
    "total_numero_de_alojamientos_turisticos_ocupados": "ocupaciones_vt",
    "numero_de_noches_ocupadas": "pernoctaciones_vt",
    "estancia_media_alacant/alicante": "media_vt"
}

df_provincia_vt = process_df(df_provincia_vt, mapeo=mapeo_prov)

In [ ]:
# Reemplazamos la 'M' por un '-' y lo convertimos a fecha
df_provincia_vt['fecha_dt'] = pd.to_datetime(df_provincia_vt[DatasetKeys.FECHA].str.replace('M', '-'))

# Esto hace que "2024-06-30" (AMAEM) y "2024-06-01" (INE) sean ambos "2024-06"
df_amaem['mes_union'] = df_amaem[DatasetKeys.FECHA].dt.to_period('M')
df_provincia_vt['mes_union'] = df_provincia_vt['fecha_dt'].dt.to_period('M')

# Usamos 'left' para que no se borre nada de AMAEM
df_resultado = pd.merge(
    df_amaem, 
    df_provincia_vt.drop(columns=[DatasetKeys.FECHA, 'fecha_dt']), # Quitamos las fechas viejas del INE para no duplicar
    on='mes_union', 
    how='left'
)

df_resultado = df_resultado.drop(columns=['mes_union'])

In [ ]:
display(df_resultado.drop_duplicates(subset=[DatasetKeys.FECHA]).head(12))

### Hoteles

In [ ]:
df_provincia_hotel

###

## Comunidad

In [ ]:
params_com = {"encoding": "latin1", "sep": ";"}

# Cargar archivos de Comunidad
df_comunidad_tipo_aloj = pd.read_csv(Paths.INE_COMUNIDAD_TIPO_ALOJ, **params_com)
df_comunidad_total     = pd.read_csv(Paths.INE_COMUNIDAD_TOTAL, **params_com)


In [ ]:
mapeo_comunidad = {
    'destino_principal': 'destino',
    'concepto_turistico': 'concepto',
    'alojamiento_principal_nivel_1': 'aloj_n1',
    'alojamiento_principal_nivel_2': 'aloj_n2',
    'motivo_principal': 'motivo',
    'transporte_principal': 'transporte',
    'forma_de_organizacion_del_viaje': 'organizacion',
    'duracion_del_viaje': 'duracion',
    'comunidad_autonoma_de_residencia': 'residencia',
    'periodo': 'periodo',
    'total': 'total'
}

drop_comunidad = ['destino', 'motivo', 'transporte', 'organizacion', 'duracion', 'residencia']

In [ ]:
df_comunidad_tipo_aloj = process_df(df_comunidad_total, mapeo=mapeo_comunidad, drop=drop_comunidad)
df_comunidad_total     = process_df(df_comunidad_total, mapeo=mapeo_comunidad, drop=drop_comunidad)

In [ ]:
df_comunidad_tipo_aloj.head(10)

In [ ]:
df_comunidad_total